In [1]:
from pathlib import Path
import os
import sys

ROOT = Path.cwd()
if not (ROOT / "data" / "manifests").exists():
    ROOT = Path(r"D:\\FILE AND TASK\\TA")
assert (ROOT / "data" / "manifests").exists(), ROOT
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
os.environ["MPLBACKEND"] = "Agg"

try:
    from IPython.display import display
except ImportError:
    def display(item):
        print(item)

import json
import numpy as np
import pandas as pd
import librosa
import matplotlib.pyplot as plt

print("Project:", ROOT)
print("Python:", sys.executable)
print("NumPy:", np.__version__)
print("Pandas:", pd.__version__)
print("Librosa:", librosa.__version__)


Project: D:\FILE AND TASK\TA
Python: c:\Users\Fabio\AppData\Local\Programs\Python\Python314\python.exe
NumPy: 2.5.2
Pandas: 3.0.5
Librosa: 1.0.0


In [2]:
# Kode run_e0_preprocessing_verification.py ditempatkan langsung di notebook ini.
# Entry point CLI tidak dipanggil otomatis agar hasil valid tidak tertimpa.
__file__ = str(ROOT / "scripts" / "run_e0_preprocessing_verification.py")

"""Run the formal E0 preprocessing audit without writing to the paper.

E0 is a deterministic 100-recording audit of the active Indonesia split. It
is not a retrieval experiment. Every output is self-contained beneath
``results/reproductions/E0_<UTC timestamp>``.
"""


import hashlib
import json
import os
import subprocess
import sys
from datetime import datetime, timezone
from pathlib import Path

PROJECT_ROOT = Path(__file__).resolve().parent.parent
os.environ.setdefault("MPLCONFIGDIR", str(PROJECT_ROOT / "results" / ".mplconfig"))
os.environ["MPLBACKEND"] = "Agg"

import librosa
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import soundfile as sf

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.embeddings import AudioRepresentationExtractor
from src.preprocess import DURATION_SEC, TARGET_RMS, TARGET_SAMPLES, TARGET_SR, compute_rms, preprocess_audio

MANIFEST_PATH = PROJECT_ROOT / "data" / "manifests" / "dataset_split.csv"
OUTPUT_ROOT = PROJECT_ROOT / "results" / "reproductions"
SAMPLE_SEED = 20261012
SAMPLE_SIZE = 100


def sha256_file(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as source:
        for block in iter(lambda: source.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()


def git_sha() -> str | None:
    try:
        return subprocess.check_output(
            ["git", "rev-parse", "HEAD"], cwd=PROJECT_ROOT, text=True, stderr=subprocess.DEVNULL
        ).strip()
    except (OSError, subprocess.CalledProcessError):
        return None


def select_window_start(resampled_mono: np.ndarray) -> int:
    """Return the exact start index used by the current preprocessing algorithm."""
    if len(resampled_mono) <= TARGET_SAMPLES:
        return 0
    hop = TARGET_SAMPLES // 4
    count = max(1, (len(resampled_mono) - TARGET_SAMPLES) // hop + 1)
    rms_values = [compute_rms(resampled_mono[i * hop : i * hop + TARGET_SAMPLES]) for i in range(count)]
    return int(np.argmax(rms_values) * hop)


def deterministic_sample(manifest: pd.DataFrame) -> pd.DataFrame:
    if len(manifest) < SAMPLE_SIZE:
        raise ValueError(f"Active manifest has only {len(manifest)} rows; E0 needs {SAMPLE_SIZE}.")
    pieces: list[pd.DataFrame] = []
    remaining = SAMPLE_SIZE
    roles = [(role, group.reset_index(drop=True)) for role, group in manifest.groupby("split_role", sort=True)]
    for index, (_, group) in enumerate(roles):
        take = round(SAMPLE_SIZE * len(group) / len(manifest)) if index < len(roles) - 1 else remaining
        take = min(take, len(group))
        remaining -= take
        pieces.append(group.sample(n=take, random_state=SAMPLE_SEED + index))
    sample = pd.concat(pieces, ignore_index=True)
    if len(sample) != SAMPLE_SIZE:
        raise AssertionError(f"Internal E0 sampling error: {len(sample)} rows selected.")
    return sample.sort_values(["split_role", "recording_id"]).reset_index(drop=True)


def inspect_recording(row: pd.Series) -> dict[str, object]:
    path = PROJECT_ROOT / str(row["file_path"])
    try:
        info = sf.info(path)
        container, subtype = info.format, info.subtype
        sr_native, channels, duration_native = int(info.samplerate), int(info.channels), float(info.duration)
        soundfile_error = ""
    except RuntimeError as error:
        container, subtype, sr_native, channels, duration_native, soundfile_error = "UNREADABLE_BY_SOUNDFILE", "", np.nan, np.nan, np.nan, str(error)
    decoded, _ = librosa.load(path, sr=TARGET_SR, mono=True)
    start_sample = select_window_start(decoded)
    processed = preprocess_audio(str(path))
    expected_container = path.suffix.removeprefix(".").upper()
    mismatch = bool(container not in {expected_container, "MPEG" if expected_container == "MP3" else expected_container})
    return {
        "recording_id": row["recording_id"], "species_key": row["species_key"], "split_role": row["split_role"],
        "file_path": row["file_path"], "extension": path.suffix.lower(), "container": container, "subtype": subtype,
        "extension_container_mismatch": mismatch, "soundfile_error": soundfile_error,
        "native_sample_rate_hz": sr_native, "native_channels": channels, "native_duration_seconds": duration_native,
        "decoded_sample_rate_hz": TARGET_SR, "decoded_samples_before_window": int(len(decoded)),
        "selected_window_start_sample": start_sample, "selected_window_start_seconds": start_sample / TARGET_SR,
        "selected_window_duration_seconds": DURATION_SEC, "processed_samples": int(len(processed)),
        "processed_sample_rate_hz": TARGET_SR, "processed_rms": compute_rms(processed),
        "processed_peak": float(np.max(np.abs(processed))), "preprocess_length_ok": bool(len(processed) == TARGET_SAMPLES),
        "preprocess_finite_ok": bool(np.isfinite(processed).all()),
    }


def make_plots(audit: pd.DataFrame, figures_dir: Path) -> None:
    fig, axes = plt.subplots(2, 2, figsize=(13, 8))
    audit["native_duration_seconds"].dropna().plot.hist(bins=24, ax=axes[0, 0], color="#356f9b")
    axes[0, 0].set(title="Native duration", xlabel="seconds")
    audit["native_sample_rate_hz"].dropna().value_counts().sort_index().plot.bar(ax=axes[0, 1], color="#5a9d6f")
    axes[0, 1].set(title="Native sampling rate", xlabel="Hz")
    audit["selected_window_start_seconds"].plot.hist(bins=24, ax=axes[1, 0], color="#a36c3e")
    axes[1, 0].set(title="Selected 5-second window start", xlabel="seconds")
    audit["container"].value_counts().plot.bar(ax=axes[1, 1], color="#86589c")
    axes[1, 1].set(title="Decoded container", xlabel="container")
    fig.suptitle("E0 formal preprocessing audit (100 deterministic active-split recordings)")
    fig.tight_layout()
    fig.savefig(figures_dir / "e0_audio_properties.png", dpi=180)
    plt.close(fig)


def main() -> Path:
    manifest = pd.read_csv(MANIFEST_PATH)
    sample = deterministic_sample(manifest)
    now = datetime.now(timezone.utc)
    output_dir = OUTPUT_ROOT / f"E0_{now.strftime('%Y%m%d_%H%M%S')}"
    output_dir.mkdir(parents=True, exist_ok=False)
    tables_dir = output_dir / "tables"
    figures_dir = output_dir / "figures"
    tables_dir.mkdir()
    figures_dir.mkdir()
    print(f"E0 formal: auditing {len(sample)} active recordings; output: {output_dir}")
    records = []
    for position, (_, row) in enumerate(sample.iterrows(), start=1):
        records.append(inspect_recording(row))
        if position % 10 == 0 or position == len(sample):
            print(f"E0 audio audit: {position}/{len(sample)}", flush=True)
    audit = pd.DataFrame(records)
    audit.to_csv(tables_dir / "per_recording.csv", index=False)
    probe_row = sample.iloc[0]
    probe_audio = preprocess_audio(str(PROJECT_ROOT / probe_row["file_path"]))
    probes = []
    for code, dimension in (("R0", 40), ("R1", 2048), ("R2", 1024), ("R3", 40)):
        embedding = AudioRepresentationExtractor(code).extract(probe_audio, item_id=f"e0_probe:{probe_row['recording_id']}")
        probes.append({"representation": code, "expected_dimension": dimension, "actual_dimension": int(embedding.size), "finite": bool(np.isfinite(embedding).all()), "l2_norm": float(np.linalg.norm(embedding))})
    pd.DataFrame(probes).to_csv(tables_dir / "representation_probe.csv", index=False)
    summary = pd.DataFrame([
        {"metric": "audit_rows", "value": len(audit)}, {"metric": "unique_species", "value": int(audit["species_key"].nunique())},
        {"metric": "native_duration_min_seconds", "value": float(audit["native_duration_seconds"].min())},
        {"metric": "native_duration_median_seconds", "value": float(audit["native_duration_seconds"].median())},
        {"metric": "native_duration_max_seconds", "value": float(audit["native_duration_seconds"].max())},
        {"metric": "extension_container_mismatches", "value": int(audit["extension_container_mismatch"].sum())},
        {"metric": "soundfile_decode_errors", "value": int(audit["soundfile_error"].ne("").sum())},
        {"metric": "preprocess_length_failures", "value": int((~audit["preprocess_length_ok"]).sum())},
        {"metric": "preprocess_nonfinite_failures", "value": int((~audit["preprocess_finite_ok"]).sum())},
        {"metric": "target_rms", "value": TARGET_RMS},
    ])
    summary.to_csv(tables_dir / "summary.csv", index=False)
    make_plots(audit, figures_dir)
    metadata = {
        "artifact_schema": "e0_preprocessing_audit_v1", "status": "complete", "timestamp_utc": now.isoformat(),
        "git_sha": git_sha(), "manifest_sha256": sha256_file(MANIFEST_PATH),
        "preprocess_code_sha256": sha256_file(PROJECT_ROOT / "src" / "preprocess.py"),
        "sample_seed": SAMPLE_SEED, "sample_size": SAMPLE_SIZE,
        "sample_roles": {str(k): int(v) for k, v in sample["split_role"].value_counts().sort_index().items()},
        "protocol": {"target_sample_rate_hz": TARGET_SR, "target_samples": TARGET_SAMPLES, "selected_window_seconds": DURATION_SEC, "target_rms": TARGET_RMS},
        "representation_probe_recording_id": str(probe_row["recording_id"]),
    }
    (output_dir / "metadata.json").write_text(json.dumps(metadata, indent=2) + "\n", encoding="utf-8")
    (output_dir / "README.md").write_text(
        "# E0 — Audit pra-pemrosesan\n\n"
        "- `tables/summary.csv`: ringkasan audit.\n"
        "- `tables/per_recording.csv`: detail 100 rekaman.\n"
        "- `tables/representation_probe.csv`: cek R0–R3.\n"
        "- `figures/e0_audio_properties.png`: gambar audit.\n",
        encoding="utf-8",
    )
    print(f"E0 complete: {output_dir}")
    return output_dir

print("Kode run_e0_preprocessing_verification.py telah dimuat di notebook.")


Kode run_e0_preprocessing_verification.py telah dimuat di notebook.


In [3]:
run = ROOT / "results/reproductions/E0_20261011_202557"
assert run.is_dir(), run
metadata = json.loads((run / "metadata.json").read_text(encoding="utf-8"))
print(json.dumps(metadata, ensure_ascii=False, indent=2))
summary = pd.read_csv(run / "tables/summary.csv")
probe = pd.read_csv(run / "tables/representation_probe.csv")
per_recording = pd.read_csv(run / "tables/per_recording.csv")
display(summary)
display(probe)
print("E0 memeriksa 100 rekaman aktif secara deterministik. E0 adalah audit preprocessing dan bukan evaluasi retrieval.")


{
  "artifact_schema": "e0_preprocessing_audit_v1",
  "status": "complete",
  "timestamp_utc": "2026-10-11T20:25:57.534313+00:00",
  "git_sha": "930f6f2ea381221ba29fa09227e86d3d3df1eb33",
  "manifest_sha256": "7808cdf057f135aaa193f47e27415318edf46c862c09dee4c7394bc5760e8ed8",
  "preprocess_code_sha256": "6c01850783d294f31a02a915d0497046c6903db7e11c81dc63330b94f982da08",
  "sample_seed": 20261012,
  "sample_size": 100,
  "sample_roles": {
    "calibration": 19,
    "gallery": 55,
    "query_clean": 26
  },
  "protocol": {
    "target_sample_rate_hz": 32000,
    "target_samples": 160000,
    "selected_window_seconds": 5.0,
    "target_rms": 0.05
  },
  "representation_probe_recording_id": "XC1076576"
}


,metric,value
0,audit_rows,100.000000
1,unique_species,20.000000
2,native_duration_min_seconds,4.477458
3,native_duration_median_seconds,31.864816
4,native_duration_max_seconds,199.152646
5,extension_container_mismatches,6.000000
6,soundfile_decode_errors,0.000000
7,preprocess_length_failures,0.000000
8,preprocess_nonfinite_failures,0.000000
9,target_rms,0.050000


,representation,expected_dimension,actual_dimension,finite,l2_norm
0,R0,40,40,True,1.0
1,R1,2048,2048,True,1.0
2,R2,1024,1024,True,1.0
3,R3,40,40,True,1.0


E0 memeriksa 100 rekaman aktif secara deterministik. E0 adalah audit preprocessing dan bukan evaluasi retrieval.


In [4]:
fig = plt.imread(run / "figures/e0_audio_properties.png")
plt.figure(figsize=(13, 8))
plt.imshow(fig)
plt.axis("off")
plt.show()
display(per_recording.head(20))
print("Seluruh 100 baris audit tersedia di:", run / "tables/per_recording.csv")


C:\Users\Fabio\AppData\Local\Temp\ipykernel_10536\794801059.py:5: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


,recording_id,species_key,split_role,file_path,extension,container,subtype,extension_container_mismatch,soundfile_error,native_sample_rate_hz,...,decoded_samples_before_window,selected_window_start_sample,selected_window_start_seconds,selected_window_duration_seconds,processed_samples,processed_sample_rate_hz,processed_rms,processed_peak,preprocess_length_ok,preprocess_finite_ok
0,XC1076576,moutai1,calibration,data/xeno_canto_indonesia/moutai1/XC1076576.mp3,.mp3,MP3,MPEG_LAYER_III,False,NaN,48000,...,3128881,1240000,38.75,5.0,160000,32000,0.050000,0.531084,True,True
1,XC20588,gpipig1,calibration,data/xeno_canto_indonesia/gpipig1/XC20588.mp3,.mp3,MP3,MPEG_LAYER_III,False,NaN,44100,...,490685,0,0.00,5.0,160000,32000,0.050000,0.263124,True,True
2,XC214970,pygcup1,calibration,data/xeno_canto_indonesia/pygcup1/XC214970.mp3,.mp3,MP3,MPEG_LAYER_III,False,NaN,48000,...,766490,360000,11.25,5.0,160000,32000,0.050000,0.309528,True,True
3,XC215016,lessho1,calibration,data/xeno_canto_indonesia/lessho1/XC215016.mp3,.mp3,MP3,MPEG_LAYER_III,False,NaN,48000,...,189175,0,0.00,5.0,160000,32000,0.050000,0.293806,True,True
4,XC268523,rubcuc1,calibration,data/xeno_canto_indonesia/rubcuc1/XC268523.mp3,.mp3,MP3,MPEG_LAYER_III,False,NaN,44100,...,2099456,1840000,57.50,5.0,160000,32000,0.050000,0.286935,True,True
5,XC298647,rubcuc1,calibration,data/xeno_canto_indonesia/rubcuc1/XC298647.mp3,.mp3,MP3,MPEG_LAYER_III,False,NaN,44100,...,2316971,120000,3.75,5.0,160000,32000,0.050000,0.230134,True,True
6,XC31430,pygcup1,calibration,data/xeno_canto_indonesia/pygcup1/XC31430.mp3,.mp3,MP3,MPEG_LAYER_III,False,NaN,44100,...,381306,0,0.00,5.0,160000,32000,0.050000,0.242296,True,True
7,XC31431,pygcup1,calibration,data/xeno_canto_indonesia/pygcup1/XC31431.mp3,.mp3,MP3,MPEG_LAYER_III,False,NaN,44100,...,355200,80000,2.50,5.0,160000,32000,0.050000,0.263824,True,True
8,XC336108,pygcup1,calibration,data/xeno_canto_indonesia/pygcup1/XC336108.mp3,.mp3,MP3,MPEG_LAYER_III,False,NaN,44100,...,667992,120000,3.75,5.0,160000,32000,0.050000,0.532881,True,True
9,XC376482,comior1,calibration,data/xeno_canto_indonesia/comior1/XC376482.mp3,.mp3,MP3,MPEG_LAYER_III,False,NaN,44100,...,733101,40000,1.25,5.0,160000,32000,0.050000,0.378500,True,True


Seluruh 100 baris audit tersedia di: D:\FILE AND TASK\TA\results\reproductions\E0_20261011_202557\tables\per_recording.csv
